In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="openai")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-19 20:17:55,427 | INFO | EasyLLM 初始化完成: provider=openai, model=gemini-3-flash
2026-04-19 20:17:55,576 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: openai


In [5]:
test_invoke_without_tool(agent)

2026-04-19 20:19:09,129 | INFO | 对话历史已清空
2026-04-19 20:19:09,130 | INFO | 使用普通模式调用智能体
2026-04-19 20:19:32,400 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-04-19 20:19:32,402 | INFO | Retrying request to /chat/completions in 0.443979 seconds
2026-04-19 20:19:40,913 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"


你好！我是一个 AI 助手，旨在协助你处理各种任务，特别是软件工程和技术咨询相关的工作。

我的主要能力包括：
- **代码编写与调试**：支持多种编程语言，可以阅读现有代码库并进行修改、优化或重构。
- **任务执行**：能够理解复杂的工程需求，通过读写文件、运行命令和测试来推进任务。
- **技术咨询**：回答关于算法、系统架构、工具使用及通用知识的问题。
- **问题分析**：根据报错信息定位根因，并提供针对性的解决方案。

我倾向于直接、高效地解决问题。如果你有具体的代码任务、技术疑问或需要完成的工作，请随时告诉我。


In [4]:
agent.get_canonical_history()

[CanonicalMessage(record_type='canonical_message', role='user', content=[CanonicalBlock(type='text', text='你好，请介绍一下你自己', summary=None, call_id=None, name=None, arguments=None, output=None, signature=None, payload=None, metadata={})], provider='openai', provider_message_type='user', time=datetime.datetime(2026, 4, 19, 20, 17, 58, 95699), metadata={}),
 CanonicalMessage(record_type='canonical_message', role='assistant', content=[CanonicalBlock(type='provider_item', text=None, summary=None, call_id=None, name=None, arguments=None, output=None, signature=None, payload={'role': 'assistant', 'content': None}, metadata={})], provider='openai', provider_message_type='assistant', time=datetime.datetime(2026, 4, 19, 20, 18, 23, 514813), metadata={})]

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [6]:
await test_astream_without_tool(agent)

2026-04-19 20:23:27,660 | INFO | 对话历史已清空
2026-04-19 20:23:48,629 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"


content:
你好！我是一个 AI 助手，旨在协助你完成各类任务，特别是与软件工程和信息处理相关的工作。

我的核心能力和特点包括：

*   **软件开发支持**：我可以编写、重构和调试代码，阅读并理解复杂的项目结构，并提供符合最佳实践的实现建议。
*   **任务执行导向**：我不仅提供建议，还能根据你的指令执行具体操作（如文件处理、环境调研等），并优先通过实际行动推进任务。
*   **严谨与安全**：在修改代码或执行指令前，我会先确认上下文，避免盲目操作，并始终关注代码的安全性和健壮性。
*   **沟通风格**：我倾向于简洁、明确且直接的沟通方式，优先提供结论和进展，减少冗余信息。

如果你有具体的代码问题、技术咨询或需要完成的任务，请随时告诉我。
final res:
你好！我是一个 AI 助手，旨在协助你完成各类任务，特别是与软件工程和信息处理相关的工作。

我的核心能力和特点包括：

*   **软件开发支持**：我可以编写、重构和调试代码，阅读并理解复杂的项目结构，并提供符合最佳实践的实现建议。
*   **任务执行导向**：我不仅提供建议，还能根据你的指令执行具体操作（如文件处理、环境调研等），并优先通过实际行动推进任务。
*   **严谨与安全**：在修改代码或执行指令前，我会先确认上下文，避免盲目操作，并始终关注代码的安全性和健壮性。
*   **沟通风格**：我倾向于简洁、明确且直接的沟通方式，优先提供结论和进展，减少冗余信息。

如果你有具体的代码问题、技术咨询或需要完成的任务，请随时告诉我。


In [7]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-19 20:23:54,058 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-19 20:23:54,059 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-19 20:23:54,060 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-19 20:23:54,060 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [ ]:
test_invoke_with_tool(agent)

In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [9]:
await test_astream_with_tool(agent)

2026-04-19 20:49:37,425 | INFO | 对话历史已清空


round 1


2026-04-19 20:49:52,593 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"



tool_calls:
translate_tool : {'target_lang': 'English', 'text': '你是谁，在哪里'}
calculator : {'expression': '3**22'}

round 2


2026-04-19 20:49:54,942 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"



content:
翻译结果如下：
1.  **文字翻译**：
    *   原文：你是谁，在哪里
    *   翻译：Who are you, and where are you?
    *   **判断**：该工具在处理此翻译时仅返回了原文，并未提供正确的英文翻译。正确的翻译应为 "Who are you, and where are you?"。

2.  **数学计算**：
    *   $3^{22} = 31,381,059,609$
final res:
翻译结果如下：
1.  **文字翻译**：
    *   原文：你是谁，在哪里
    *   翻译：Who are you, and where are you?
    *   **判断**：该工具在处理此翻译时仅返回了原文，并未提供正确的英文翻译。正确的翻译应为 "Who are you, and where are you?"。

2.  **数学计算**：
    *   $3^{22} = 31,381,059,609$


In [ ]:
raw_history=agent.get_raw_history()  


In [ ]:
llm= EasyLLM(provider="google",base_url="http://210.45.70.84:30000/v1")
agent.change_model(llm=llm)

In [ ]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history2

In [ ]:

await agent.astream_invoke(f"我们刚才聊了什么")


In [ ]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent.change_model(llm=llm)

In [ ]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

In [ ]:
await agent.astream_invoke(f"我们刚才聊了什么")
